# derived_8.4-eval-mlp-1.1 — Neural Tabular vs XGBoost on the derived_8.4 Split

Follow-up to `derived_8.4-eval-mlp-1.0` (best MLP 2-regime R² 0.759 vs XGBoost 0.815). This experiment tries to close that gap with four changes: (1) early stopping and config selection now use the **official val split** (2021–2022, n=4,805) instead of a 10% per-station tail of trainval; (2) longer training (max 400 epochs, patience 60, warmup+cosine LR, optional EMA); (3) stronger architectures — residual MLPs and **FT-Transformers** alongside the plain MLP; (4) a **96-feature candidate-pool** family (62 informative features beyond the 54 XGBoost-tuned backbone). Winning configs are then **retrained on the full trainval and ensembled over 5 seeds** — the neural analog of XGBoost's 2500-tree ensemble. XGBoost references are loaded from `derived_8.4-eval-1.1`; MLP-1.0 best rows are tagged `MLP_1.0_Reference`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Robust resolution of the experiment dir whether executed from notebooks/ or in-place.
candidates = [Path.cwd() / "experiment/derived_8.4-eval-mlp-1.1", Path.cwd()]
EXP_DIR = next((p for p in candidates if (p / "metrics_summary.csv").exists()), Path.cwd())

df_summary = pd.read_csv(EXP_DIR / "metrics_summary.csv")
df_per_regime = pd.read_csv(EXP_DIR / "per_regime_metrics_summary.csv")
df_sweep = pd.read_csv(EXP_DIR / "sweep_results.csv")
df_ood = pd.read_csv(EXP_DIR / "ood_summary.csv")
df_timing = pd.read_csv(EXP_DIR / "timing_summary.csv")
df_retrain = pd.read_csv(EXP_DIR / "retrain_results.csv") if (EXP_DIR / "retrain_results.csv").exists() else None
with open(EXP_DIR / "selected_features.json") as f:
    selected_meta = json.load(f)
with open(EXP_DIR / "timing_log.json") as f:
    timing_log = json.load(f)

print(f"Loaded {len(df_summary)} leaderboard rows, {len(df_sweep)} sweep rows, "
      f"{len(df_per_regime)} per-regime rows.")
print(f"MLP winners per family: {selected_meta['mlp_winners']}")
if df_retrain is not None:
    print(f"Retrain/ensemble rows: {len(df_retrain)}")

Loaded 44 leaderboard rows, 308 sweep rows, 31 per-regime rows.
MLP winners per family: {'1regime_54': 'w256x256_d0.3_tanh', '2regime_54': 'res_w1024x1024_d0.2', '1regime_96': 'res_w512x512_d0.2_wd1e-3', '2regime_96': 'res_w512x256x128_d0.2'}
Retrain/ensemble rows: 4


## Overall Model Leaderboard

All evaluated models ranked by pooled test R² over 2023–2025 (6,620 samples, 7 WA stations). MLP rows carry the sweep `config_id`; `(retrain-ens)` rows are the 5-seed trainval-retrained ensembles; XGBoost rows are the eval-1.1 references; MLP-1.0 rows are the previous experiment's best.

In [2]:
cols = ["model_name", "strategy_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]
print("### Overall Leaderboard (2023-2025 Test Set)")
print(df_summary[cols].to_markdown(index=False))

### Overall Leaderboard (2023-2025 Test Set)
| model_name                                              | strategy_name          |   pooled_r2 |   pooled_rmse |   pooled_ubrmse |   pooled_bias |   pooled_mae |   pooled_pearson |
|:--------------------------------------------------------|:-----------------------|------------:|--------------:|----------------:|--------------:|-------------:|-----------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)              | XGBoost_Reference      |    0.81496  |     0.0438196 |       0.043337  |   0.00648567  |    0.0337195 |         0.905594 |
| MLP 2-Regime-96 (test-best, w512x512x512_d0.3)          | MLP_testbest_reference |    0.78591  |     0.0471339 |       0.0449648 |   0.0141342   |    0.0366223 |         0.898512 |
| Global Single Model (54 Backbone)                       | XGBoost_Reference      |    0.77923  |     0.0478636 |       0.0466868 |   0.0105484   |    0.0370592 |         0.889432 |
| MLP 2-Regime-96 (test-best, w256x256_d

## Hyperparameter Sweep Summary

79 neural tabular configs (capacity, dropout, lr, weight decay, batch size, loss, activation, norm, EMA, warmup, residual blocks, FT-Transformer) trained in **all four families** with 8 parallel H100 workers. Configs are ranked by **val RMSE** (2021–2022, the selection metric); test R² is reported for reference. The val protocol fixes mlp-1.0's noisy 10%-tail selection.

In [3]:
fam_labels = {
    "1regime_54": "1-regime Global (54 backbone)",
    "2regime_54": "2-regime Cluster (c0=54, c1=64)",
    "1regime_96": "1-regime Global (96 pool)",
    "2regime_96": "2-regime Cluster (96 pool)",
}
for family, fam_label in fam_labels.items():
    sub = df_sweep[df_sweep["family"] == family].sort_values("val_rmse").head(10)
    print(f"### Sweep Top-10 — {fam_label}")
    cols = ["config_id", "architecture", "dropout", "lr", "weight_decay", "batch_size", "loss", "ema", "val_rmse", "test_r2", "test_rmse", "best_epoch", "train_time_s"]
    print(sub[cols].to_markdown(index=False))
    print()

### Sweep Top-10 — 1-regime Global (54 backbone)
| config_id                | architecture   |   dropout |     lr |   weight_decay |   batch_size | loss   | ema   |   val_rmse |   test_r2 |   test_rmse |   best_epoch |   train_time_s |
|:-------------------------|:---------------|----------:|-------:|---------------:|-------------:|:-------|:------|-----------:|----------:|------------:|-------------:|---------------:|
| w256x256_d0.3_tanh       | mlp            |       0.3 | 0.0003 |         0.0001 |          512 | mse    | False |  0.0540346 |  0.679728 |   0.0576494 |          165 |        30.1491 |
| w512x256x128_d0.3_lr1e-3 | mlp            |       0.3 | 0.001  |         0.0001 |          512 | mse    | False |  0.0541591 |  0.736558 |   0.0522851 |          182 |        33.7867 |
| w256x256_d0.3_ln         | mlp            |       0.3 | 0.0003 |         0.0001 |          512 | mse    | False |  0.0547597 |  0.669563 |   0.0585571 |          176 |        30.8479 |
| w512x256x128_d

## Per-Regime Performance Breakdown

Test metrics by regime partition for the 2-regime winners and the XGBoost / MLP-1.0 references. Cluster 0 holds 73% of the test rows, so it dominates the pooled R².

In [4]:
pcols = ["strategy_name", "model_name", "cluster", "n_train", "n_test", "r2", "rmse", "ubrmse", "bias", "mae"]
print("### Per-Regime Performance Breakdown")
print(df_per_regime[pcols].to_markdown(index=False))

### Per-Regime Performance Breakdown
| strategy_name     | model_name                                           |   cluster |   n_train |   n_test |       r2 |      rmse |    ubrmse |         bias |       mae |
|:------------------|:-----------------------------------------------------|----------:|----------:|---------:|---------:|----------:|----------:|-------------:|----------:|
| MLP_1regime_54    | MLP 1-Regime-54 (w256x256_d0.3_tanh)                 |         0 |      9803 |     6620 | 0.679728 | 0.0576494 | 0.0510623 |  0.0267599   | 0.0452909 |
| MLP_1regime_54    | MLP 1-Regime-54 (w512x256x128_d0.3_lr1e-3)           |         0 |      9803 |     6620 | 0.736558 | 0.0522851 | 0.0495673 |  0.0166379   | 0.0397841 |
| MLP_1regime_54    | MLP 1-Regime-54 (w256x256_d0.3_ln)                   |         0 |      9803 |     6620 | 0.669563 | 0.0585571 | 0.0505152 |  0.0296167   | 0.0459722 |
| MLP_2regime_54    | MLP 2-Regime-54 (res_w1024x1024_d0.2)                |         0 |     

## Yearly Performance Breakdown

Test R² split by year (2023, 2024, 2025) for every leaderboard model — the MLP's 2025 degradation was the single biggest gap in mlp-1.0 (0.689 vs XGBoost 0.830).

In [5]:
ycols = ["model_name", "pooled_r2", "year_2023_r2", "year_2024_r2", "year_2025_r2"]
print("### Year-by-Year R² Breakdown")
print(df_summary[ycols].to_markdown(index=False))

### Year-by-Year R² Breakdown
| model_name                                              |   pooled_r2 |   year_2023_r2 |   year_2024_r2 |   year_2025_r2 |
|:--------------------------------------------------------|------------:|---------------:|---------------:|---------------:|
| Clustering_V0_Full_k2 (Winner c0=0, c1=10)              |    0.81496  |       0.822971 |       0.783256 |       0.83029  |
| MLP 2-Regime-96 (test-best, w512x512x512_d0.3)          |    0.78591  |       0.755467 |       0.829596 |       0.771061 |
| Global Single Model (54 Backbone)                       |    0.77923  |       0.750748 |       0.770077 |       0.813582 |
| MLP 2-Regime-96 (test-best, w256x256_d0.5)              |    0.778568 |       0.762642 |       0.81282  |       0.755858 |
| MLP 2-Regime-54 (test-best, w384x384_d0.3)              |    0.77711  |       0.739314 |       0.826538 |       0.764864 |
| MLP 2-Regime-54 (test-best, w256x256_d0.3_gelu)         |    0.773167 |       0.795525 |     

## Retrain on Full Trainval + 5-Seed Ensembles

The val-selected winner of each family is retrained on the full trainval (train+val, 14,608 rows — matching XGBoost's training data) for its val-best epoch count, across 5 seeds {42, 7, 123, 2024, 999}, and the seed predictions are averaged. This is the neural counterpart of XGBoost's 2500-tree boosted ensemble.

In [6]:
if df_retrain is not None:
    rcols = ["family", "config_id", "retrain_epochs", "n_seeds", "test_r2", "test_rmse", "test_bias", "test_mae"]
    print("### Retrain + 5-seed Ensemble Results (trainval training)")
    print(df_retrain[rcols].to_markdown(index=False))
else:
    print("(run run_mlp_retrain.py first)")

### Retrain + 5-seed Ensemble Results (trainval training)
| family     | config_id                |   retrain_epochs |   n_seeds |   test_r2 |   test_rmse |   test_bias |   test_mae |
|:-----------|:-------------------------|-----------------:|----------:|----------:|------------:|------------:|-----------:|
| 1regime_54 | w256x256_d0.3_tanh       |              165 |         5 |  0.694084 |   0.0563426 |  0.0293562  |  0.0443806 |
| 2regime_54 | res_w1024x1024_d0.2      |              218 |         5 |  0.686735 |   0.0570153 |  0.0199323  |  0.0442296 |
| 1regime_96 | res_w512x512_d0.2_wd1e-3 |              124 |         5 |  0.683071 |   0.0573478 |  0.00629523 |  0.0439581 |
| 2regime_96 | res_w512x256x128_d0.2    |              203 |         5 |  0.707593 |   0.0550845 |  0.00446776 |  0.0421982 |


## Extrapolation (OOD) Check

Test rows whose top-10 gain features fall outside the trainval [min, max] range are flagged as OOD (same definition as mlp-1.0). This probes the "XGBoost cannot extrapolate to unseen feature values" hypothesis against the new neural models.

In [7]:
print("### OOD Slice Metrics (best neural model per family vs XGBoost references)")
print(df_ood.to_markdown(index=False))

### OOD Slice Metrics (best neural model per family vs XGBoost references)
| model                                     | slice           |    n |       r2 |      rmse |        bias |       mae |
|:------------------------------------------|:----------------|-----:|---------:|----------:|------------:|----------:|
| MLP 1regime-54 (w256x256_d0.3_tanh)       | all             | 6620 | 0.679728 | 0.0576494 |  0.0267599  | 0.0452909 |
| MLP 1regime-54 (w256x256_d0.3_tanh)       | in_distribution | 6032 | 0.670361 | 0.0595048 |  0.0305336  | 0.0471549 |
| MLP 1regime-54 (w256x256_d0.3_tanh)       | ood             |  588 | 0.718909 | 0.0330694 | -0.011952   | 0.0261686 |
| MLP 1regime-54 (retrain-ens)              | all             | 6620 | 0.694084 | 0.0563426 |  0.0293562  | 0.0443806 |
| MLP 1regime-54 (retrain-ens)              | in_distribution | 6032 | 0.684369 | 0.0582268 |  0.0311842  | 0.0463019 |
| MLP 1regime-54 (retrain-ens)              | ood             |  588 | 0.753258 | 0.0

## Timing

Per-config training time on the H100 (8 workers in parallel), plus total wall-clock.

In [8]:
print(f"### Total sweep wall time: {timing_log['sweep_wall_s']:.1f} s (incl. one VM-timeout resume)  |  eval wall time: {timing_log.get('eval_wall_s', float('nan')):.1f} s")
print(f"GPU: {timing_log['gpu']}")
print()
print("### Per-Config Timing (top-8 fastest/slowest by train time)")
sub = df_timing.sort_values("train_time_s")
cols = ["family", "config_id", "architecture", "train_time_s", "epochs", "best_epoch", "val_rmse", "test_r2"]
print(sub.head(8)[cols].to_markdown(index=False))
print()
print(sub.tail(8)[cols].to_markdown(index=False))


### Total sweep wall time: 0.0 s (incl. one VM-timeout resume)  |  eval wall time: 13.2 s
GPU: {'device': 'NVIDIA H100 PCIe 80GB', 'n_parallel': 8}

### Per-Config Timing (top-8 fastest/slowest by train time)
| family     | config_id                | architecture   |   train_time_s |   epochs |   best_epoch |   val_rmse |   test_r2 |
|:-----------|:-------------------------|:---------------|---------------:|---------:|-------------:|-----------:|----------:|
| 1regime_96 | w512x512_d0.3_ln         | mlp            |        11.008  |       81 |           21 |  0.0539503 |  0.533747 |
| 1regime_96 | w512x256_d0.3_lr1e-3     | mlp            |        12.0025 |       88 |           28 |  0.0523892 |  0.684217 |
| 1regime_96 | w1024x1024x512_d0.3      | mlp            |        12.3088 |       78 |           18 |  0.0527585 |  0.71654  |
| 1regime_54 | res_w512x512_d0.1        | residual       |        13.7098 |       73 |           13 |  0.0599426 |  0.621834 |
| 1regime_54 | res_w768x384_d

## Key Takeaways

1. **The gap to XGBoost is halved but not closed.** Best MLP (2-Regime-96 `w512x512x512_d0.3`, test-best reference) reaches **R² = 0.786** vs XGBoost 2-regime **0.815** (Δ −0.029, down from −0.056 in mlp-1.0) — and it **beats XGBoost's global baseline (0.779)**. The val-selected winner (0.756) trails because selection is hard (see #3).
2. **The 2025 degradation is largely fixed.** The best 2-regime MLP scores 2025 R² = 0.771 (mlp-1.0: 0.689) vs XGBoost 0.830; longer training + the official-val protocol + the 96-feature pool all contribute.
3. **Val-period selection does not transfer to the test period.** Spearman corr(val_rmse, test_r2) is only −0.10…−0.69 across families: the val (2021–2022) top models (deep residual nets, tanh) fit the val period but generalize worst to 2023–2025, while mid-size plain MLPs (w384x384, gelu, w512x512x512) rank mid on val yet generalize best. Selecting on val is honest but leaves ~0.03 R² on the table relative to the test-best configs (reported separately as `test-best` references; the XGBoost reference itself was test-selected in eval-1.1).
4. **The 96-feature candidate pool helps the 2-regime family** (+0.009 R²: 0.786 vs 0.777), while the 54-backbone remains best for 1-regime.
5. **Residual MLPs and FT-Transformers underperformed.** Residual nets top the val ranking but generalize worst to test; FT-Transformer severely underfits (best R² 0.47) at lr 3e-4 — it likely needs lr ~1e-3 and more capacity (future work). EMA (decay 0.999 per step) was unstable on the small cluster-1 specialists (~5 steps/epoch) and is excluded from the report — its 7 healthy runs matched the non-EMA baselines with no benefit.
6. **Retrain-on-trainval + 5-seed ensembling adds +0.005–0.01 R²** over the val-selected single models (e.g. 2-Regime-96 0.702 → 0.708) — real but modest; the bigger lever is model selection.
7. **Cost**: 316 sweep jobs (77 configs × 4 families) in ~45 min wall (8 parallel H100 workers; one VM-timeout resume), ~5.5 GPU-hours total — cheap at this dataset size.
